# 08: RadImageNet pretraining for the Phase 2 classifier

The sprint plan asked us to compare ImageNet and RadImageNet pretraining (task p2-t3). Notebook 07 uses EfficientNet-B0 pretrained on ImageNet, which is made of ordinary photos. RadImageNet is a collection of about 1.35 million CT, MRI and ultrasound images, so a model pretrained on it might suit our kidney ultrasound better. RadImageNet only publishes weights for ResNet50, DenseNet121, InceptionV3 and InceptionResNetV2, not for EfficientNet-B0, so we have to make them ourselves. That needs a large GPU, so this notebook is meant to run on the university cluster.

The notebook has two stages:

1. **Pretraining.** Train EfficientNet-B0 to recognise RadImageNet's classes. By default it starts from the ImageNet weights and trains for 30 epochs. With `INIT=scratch` it starts from random weights and trains for 90 epochs.
2. **Fine-tuning.** Repeat notebook 07's training on the 1,985 kidney crops with exactly the same folds, recipe and threshold rule. Only the starting weights change. Then compare with the ImageNet model.

**How to run it on the cluster**

1. Get RadImageNet from its authors and copy it to the cluster. Any folder layout works: the class of an image is the folder it is in.
2. Clone the repository and check out the `radimagenet-pretraining` branch. The kidney crops and folds needed for stage 2 are already in the repository.
3. Create an environment with PyTorch for the cluster's CUDA version, then: `pip install timm pandas scikit-learn scipy matplotlib tqdm jupyter nbconvert`.
4. First, a quick test with fake images (a few minutes): `SMOKE_TEST=1 jupyter nbconvert --to notebook --execute notebooks/08_radimagenet_pretraining.ipynb --output 08_smoke_test.ipynb`
5. Edit `scripts/run_radimagenet.sbatch` (account, partition, `RADIMAGENET_DIR`) and submit it with `sbatch scripts/run_radimagenet.sbatch`. The notebook saves a checkpoint after every epoch. If the job hits the time limit, submit it again and it continues where it stopped.
6. Bring back `efficientnet_b0_radimagenet_backbone.pt`, `history.csv`, `results/phase2_classifier/nb06crops_weighted/radimagenet/` and the executed notebook.

Progress is written to `results/radimagenet_pretraining/<init>_init/train.log` while the job runs.

In [ ]:
import os, json, time, math, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import timm

# Works when started from the repository root or from notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'results/eda/folds_master.csv').exists():
    ROOT = ROOT.parent
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Settings can be changed here or through environment variables in the cluster job.
SMOKE_TEST = os.environ.get('SMOKE_TEST', '0') == '1'   # tiny fake dataset, to check that everything runs
INIT = os.environ.get('INIT', 'imagenet')                # 'imagenet' or 'scratch'
assert INIT in ('imagenet', 'scratch')
PRETRAIN = dict(epochs=int(os.environ.get('EPOCHS', 30 if INIT == 'imagenet' else 90)),
                batch_size=int(os.environ.get('BATCH_SIZE', 256)), lr=1e-3, weight_decay=0.05,
                warmup_epochs=5, label_smoothing=0.1, val_fraction=0.05, seed=42)
WORKERS = int(os.environ.get('SLURM_CPUS_PER_TASK', min(16, os.cpu_count())))
DATA = Path(os.environ.get('RADIMAGENET_DIR', ROOT / 'radimagenet'))
PRETRAIN_OUT = ROOT / 'results/radimagenet_pretraining' / ('smoke_test' if SMOKE_TEST else f'{INIT}_init')

if SMOKE_TEST:
    # Three fake classes of noisy images with different brightness patterns.
    DATA = PRETRAIN_OUT / 'fake_data'
    PRETRAIN.update(epochs=2, batch_size=32, warmup_epochs=1, val_fraction=0.2)
    rng = np.random.default_rng(0)
    for c in range(3):
        (DATA / f'US/kidney/class_{c}').mkdir(parents=True, exist_ok=True)
        for i in range(60):
            a = rng.normal(80 + 50 * c, 30, (256, 256)).clip(0, 255).astype(np.uint8)
            Image.fromarray(a).save(DATA / f'US/kidney/class_{c}/{i}.png')

PRETRAIN_OUT.mkdir(parents=True, exist_ok=True)
BACKBONE_FILE = PRETRAIN_OUT / 'efficientnet_b0_radimagenet_backbone.pt'

def log(message):
    # Printed and also appended to train.log, so progress can be followed while the cluster job runs.
    line = f'{time.strftime("%Y-%m-%d %H:%M:%S")}  {message}'
    print(line, flush=True)
    with open(PRETRAIN_OUT / 'train.log', 'a') as f:
        f.write(line + '\n')

assert DATA.is_dir(), f'RadImageNet folder not found: {DATA} (set RADIMAGENET_DIR)'
log(f'device {DEVICE} ({torch.cuda.get_device_name() if DEVICE.type == "cuda" else "no GPU"}), '
    f'{WORKERS} workers, init {INIT}, settings {PRETRAIN}')

## Stage 1: pretraining on RadImageNet

### 1.1 List the images

We scan the RadImageNet folder once and save the list in `image_index.csv`, because walking through 1.35 million files takes a while and a resubmitted job should not do it again. The class of each image is its folder path, for example `US/kidney/normal`.

We keep 5% of each class aside to follow the training. RadImageNet does not tell us which images come from the same patient, so this validation accuracy can be a bit optimistic. We only use it to pick the best epoch, never as a result.

In [ ]:
index_file = PRETRAIN_OUT / 'image_index.csv'
if index_file.exists():
    index = pd.read_csv(index_file)
else:
    paths = sorted(p for p in DATA.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'})
    index = pd.DataFrame({'path': [str(p.relative_to(DATA)) for p in paths]})
    index['class_name'] = [str(Path(p).parent) for p in index.path]
    classes = sorted(index.class_name.unique())
    index['label'] = index.class_name.map({c: i for i, c in enumerate(classes)})
    val_rows = index.groupby('label').sample(frac=PRETRAIN['val_fraction'], random_state=PRETRAIN['seed']).index
    index['split'] = np.where(index.index.isin(val_rows), 'val', 'train')
    index.to_csv(index_file, index=False)

CLASSES = index.drop_duplicates('label').sort_values('label').class_name.tolist()
counts = index.class_name.value_counts()
log(f'{len(index):,} images in {len(CLASSES)} classes ({(index.split == "train").sum():,} train, {(index.split == "val").sum():,} validation); '
    f'class sizes {counts.min():,} to {counts.max():,}')

### 1.2 Images and model

We prepare the images the same way as in notebook 07: grey-scale ultrasound repeated over three channels, brightness stretched between the 1st and 99th percentile, values in [0, 1] and no ImageNet mean/std. If pretraining used a different input, the fine-tuned model would start from features made for another kind of image.

For augmentation we use a random crop (60–100% of the image) and a horizontal flip, which are the usual choices for pretraining at this scale. The output layer is a dropout plus one linear layer, like in notebook 07, but with one output per RadImageNet class.

In [ ]:
class PercentileNormalize:
    # Stretch each image's brightness between its 1st and 99th percentile (same as notebooks 05 and 07).
    def __call__(self, img):
        a = np.asarray(img.convert('RGB'), dtype=np.float32)
        lo, hi = np.percentile(a, [1, 99])
        a = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
        return Image.fromarray((a * 255).astype(np.uint8))

PRETRAIN_TF = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), PercentileNormalize(), T.ToTensor()])
PRETRAIN_EVAL_TF = T.Compose([T.Resize((224, 224)), PercentileNormalize(), T.ToTensor()])

class RadImageNetDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths, self.labels, self.transform = frame.path.tolist(), frame.label.tolist(), transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        return self.transform(Image.open(DATA / self.paths[idx]).convert('RGB')), self.labels[idx]

random.seed(PRETRAIN['seed']); np.random.seed(PRETRAIN['seed']); torch.manual_seed(PRETRAIN['seed'])
torch.backends.cudnn.benchmark = True   # faster; exact repeatability is not needed for pretraining
loader = lambda split, tf, shuffle: DataLoader(RadImageNetDataset(index[index.split == split], tf), batch_size=PRETRAIN['batch_size'],
                                               shuffle=shuffle, drop_last=shuffle, num_workers=WORKERS, pin_memory=DEVICE.type == 'cuda',
                                               persistent_workers=WORKERS > 0)
pretrain_dl, preval_dl = loader('train', PRETRAIN_TF, True), loader('val', PRETRAIN_EVAL_TF, False)

model = timm.create_model('efficientnet_b0', pretrained=INIT == 'imagenet', num_classes=len(CLASSES))
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.classifier.in_features, len(CLASSES)))
model = model.to(DEVICE, memory_format=torch.channels_last)

### 1.3 Train

AdamW with a learning rate of 1e-3: it rises over the first 5 epochs and then follows a cosine curve down to zero. Mixed precision (AMP) roughly halves the training time on the GPU. After every epoch we save `last.pt`, with everything needed to continue, and `best.pt` when the validation accuracy improves. If `last.pt` already exists, training continues from it.

The log shows images per second after each epoch. The first epoch therefore tells us how long the whole run will take.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN['lr'], weight_decay=PRETRAIN['weight_decay'])
total_steps = PRETRAIN['epochs'] * len(pretrain_dl)
warmup_steps = PRETRAIN['warmup_epochs'] * len(pretrain_dl)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: (s + 1) / warmup_steps if s < warmup_steps
                                              else 0.5 * (1 + math.cos(math.pi * (s - warmup_steps) / max(1, total_steps - warmup_steps))))
scaler = torch.amp.GradScaler(enabled=DEVICE.type == 'cuda')
criterion = nn.CrossEntropyLoss(label_smoothing=PRETRAIN['label_smoothing'])

@torch.no_grad()
def accuracy(model, loader):
    model.eval(); correct = total = 0
    for x, y in loader:
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == 'cuda'):
            pred = model(x.to(DEVICE, memory_format=torch.channels_last)).argmax(1).cpu()
        correct += (pred == y).sum().item(); total += len(y)
    return correct / total

start_epoch, best_acc, history = 1, 0.0, []
if (PRETRAIN_OUT / 'last.pt').exists():
    state = torch.load(PRETRAIN_OUT / 'last.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler'])
    start_epoch, best_acc, history = state['epoch'] + 1, state['best_acc'], state['history']
    log(f'continuing from epoch {state["epoch"]} (best validation accuracy {best_acc:.4f})')

for epoch in range(start_epoch, PRETRAIN['epochs'] + 1):
    model.train(); started, loss_sum, seen = time.time(), 0.0, 0
    for step, (x, y) in enumerate(pretrain_dl, 1):
        x, y = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last), y.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == 'cuda'):
            loss = criterion(model(x), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        loss_sum += loss.item() * len(y); seen += len(y)
        if step % 500 == 0:
            log(f'epoch {epoch} step {step}/{len(pretrain_dl)}: loss {loss_sum / seen:.4f}, {seen / (time.time() - started):.0f} images/s')
    train_seconds = time.time() - started
    val_acc = accuracy(model, preval_dl)
    history.append(dict(epoch=epoch, train_loss=loss_sum / seen, val_accuracy=val_acc, lr=scheduler.get_last_lr()[0],
                        images_per_second=seen / train_seconds, epoch_minutes=(time.time() - started) / 60))
    log(f'epoch {epoch}/{PRETRAIN["epochs"]}: loss {loss_sum / seen:.4f}, validation accuracy {val_acc:.4f}, '
        f'{seen / train_seconds:.0f} images/s, {(time.time() - started) / 60:.1f} min')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(dict(model=model.state_dict(), epoch=epoch, val_accuracy=val_acc), PRETRAIN_OUT / 'best.pt')
    # Write to a temporary file first, so a job killed while saving cannot leave a broken last.pt.
    torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                    scaler=scaler.state_dict(), epoch=epoch, best_acc=best_acc, history=history), PRETRAIN_OUT / 'last.tmp')
    os.replace(PRETRAIN_OUT / 'last.tmp', PRETRAIN_OUT / 'last.pt')
    pd.DataFrame(history).to_csv(PRETRAIN_OUT / 'history.csv', index=False)

display(pd.DataFrame(history).set_index('epoch').round(4))

### 1.4 Save the backbone

From the best epoch we keep everything except the output layer. This *backbone* is what notebook 07 loads with `make_model(weights='radimagenet')`; the kidney classifier then gets its own new two-class output layer.

In [ ]:
best = torch.load(PRETRAIN_OUT / 'best.pt', map_location='cpu', weights_only=False)
backbone = {k: v for k, v in best['model'].items() if not k.startswith('classifier.')}
torch.save(dict(backbone_state_dict=backbone, init=INIT, best_epoch=best['epoch'], val_accuracy=best['val_accuracy'],
                classes=CLASSES, settings=PRETRAIN,
                preprocessing='RGB -> percentile 1-99 -> 224x224 -> ToTensor [0,1], no ImageNet mean/std'), BACKBONE_FILE)
log(f'saved backbone from epoch {best["epoch"]} (validation accuracy {best["val_accuracy"]:.4f}) to {BACKBONE_FILE}')

## Stage 2: fine-tuning on the kidney crops

For a fair comparison, the RadImageNet model has to be trained exactly like the ImageNet model. So instead of copying notebook 07's code, we run its setup and training cells from the notebook file itself. They load the Phase 1 crops and folds, check the splits, and define `train_cross_validation`. The only change is `weights='radimagenet'`, which makes notebook 07 start from our backbone instead of the ImageNet weights. The results go to `results/phase2_classifier/nb06crops_weighted/radimagenet/`, next to the ImageNet model's `imagenet/` folder.

In [ ]:
nb07 = json.loads((ROOT / 'notebooks/07_automatic_crop_classifier.ipynb').read_text())
for marker in ['ROOT = Path.cwd()', 'master = pd.read_csv', 'CONFIG = dict', 'def train_cross_validation', 'def threshold_table']:
    source = next(''.join(c['source']) for c in nb07['cells'] if c['cell_type'] == 'code' and marker in ''.join(c['source']))
    exec(source.split('\nfor experiment in')[0])   # the threshold cell also prints notebook 07's tables; we only need the function
RADIMAGENET_BACKBONE = BACKBONE_FILE

EXPERIMENT = 'smoke_test' if SMOKE_TEST else 'nb06crops_weighted'
if SMOKE_TEST:
    CONFIG['epochs'] = 1
if not (OUT / EXPERIMENT / 'radimagenet' / 'summary.json').exists():
    train_cross_validation(EXPERIMENT, class_weighted=True, weights='radimagenet')
display(threshold_table(EXPERIMENT, weights='radimagenet').round(3))

## Comparison with the ImageNet model

Both models are evaluated in the same way: ROC-AUC on the held-out folds, and the false negatives and false positives at the threshold we chose in notebook 07 (validation sensitivity ≥ 0.90).

Five folds give only five AUC values per model, so we also estimate how sure we can be about the difference. We resample the image clusters with replacement 2,000 times and recompute the AUC difference each time. Clusters are used rather than single images because near-duplicate images are not independent. If the 95% interval includes zero, we cannot say that one pretraining is better than the other.

In [ ]:
SENSITIVITY_TARGET = 0.90

def summarise(folder):
    oof = pd.read_csv(folder / 'oof_predictions.csv', dtype={'group_id': str})
    folds = pd.read_csv(folder / 'fold_metrics.csv')
    thresholds = {}
    for k in range(5):
        val = pd.read_csv(folder / f'inner_validation_predictions_fold_{k}.csv')
        thresholds[k] = sensitivity_threshold(val.true_label, val.probability, SENSITIVITY_TARGET)
    m = metrics(oof.true_label, oof.probability, oof.fold.map(thresholds))
    row = dict(pooled_auc=roc_auc_score(oof.true_label, oof.probability), mean_fold_auc=folds.auc.mean(),
               **{f'fold {k} auc': a for k, a in zip(folds.fold, folds.auc)},
               fn=m['fn'], fp=m['fp'], sensitivity=m['sensitivity'], specificity=m['specificity'])
    return oof, row

imagenet_oof, imagenet_row = summarise(OUT / 'nb06crops_weighted' / 'imagenet')
radimagenet_oof, radimagenet_row = summarise(OUT / EXPERIMENT / 'radimagenet')
comparison = pd.DataFrame({'ImageNet': imagenet_row, f'RadImageNet ({INIT} init)': radimagenet_row}).T
display(comparison.round(3))

both = imagenet_oof.merge(radimagenet_oof[['filename', 'probability']], on='filename', suffixes=('_imagenet', '_radimagenet'), validate='one_to_one')
rows_of_group = both.groupby('group_id').indices
groups = list(rows_of_group)
rng = np.random.default_rng(0); differences = []
for _ in range(2000):
    rows = np.concatenate([rows_of_group[g] for g in rng.choice(groups, len(groups))])
    s = both.iloc[rows]
    differences.append(roc_auc_score(s.true_label, s.probability_radimagenet) - roc_auc_score(s.true_label, s.probability_imagenet))
low, high = np.percentile(differences, [2.5, 97.5])
difference = radimagenet_row['pooled_auc'] - imagenet_row['pooled_auc']
print(f'AUC difference (RadImageNet - ImageNet): {difference:+.3f}, 95% interval {low:+.3f} to {high:+.3f}')

comparison.assign(auc_difference=[0, difference], ci_low=[np.nan, low], ci_high=[np.nan, high]).to_csv(PRETRAIN_OUT / 'comparison_with_imagenet.csv')

## Reading the result

- If the interval lies above zero, RadImageNet pretraining helps on our data. In that case we tell the team before changing anything. The Phase 3 Grad-CAM work uses the ImageNet model, so switching models affects their results.
- If the interval includes zero, the two pretrainings perform the same within what five folds can show, and we keep the ImageNet model as decided.

Either way, the interval only reflects which images happen to be in the dataset. It does not include the variation from training the network again with a different random seed.